In [ ]:
from src.config import load_env_variable
import requests
import pandas as pd
import json
from datetime import datetime
from pathlib import Path

In [ ]:
# load fmp api key
api_key = load_env_variable("FMP_API_KEY")

# set ticker
symbol = "AAPL"

In [ ]:
# api endpoint url
endpoint_url = "https://financialmodelingprep.com/stable/income-statement"

# api params
params = {
    "symbol": symbol,
    "apikey": api_key,
}

# api call
response = requests.get(url=endpoint_url, params=params, timeout=30)

# check status code
response.status_code # 200

In [ ]:
# mask api key
safe_url = response.url.replace(api_key, "REDACTED")

print(safe_url)
print(response.status_code)
print(response.text[:1000])

In [ ]:
# convert response to python json object
data = response.json()

# check type
print(type(data))

In [ ]:
# check number of records
len(data)

if len(data) == 1:
    raise ValueError("Insufficient data to calculate revenue growth CAGR")

In [ ]:
# check df shape
data[0]

In [ ]:
# convert list of records to df
df = pd.DataFrame(data)

# display df
df.head()

In [ ]:
# inspect columns
print(df.columns)

In [ ]:
# helper function returning list of columns that contain certain strings
def search_df_columns(df:pd.DataFrame, name:str | list[str]) -> list[str]:

    if isinstance(name,str):
        name = [name]

    cleaned_name = [search_term.lower().strip() for search_term in name]

    return [col for col in df.columns if any(search_term in col.lower().strip() for search_term in cleaned_name)]

search_df_columns(df, ["sym", "date", "fiscal", "per", "rev"])

In [ ]:
# inspect cols needed to calculate rev growth CAGR
print(df[['date',
'symbol',
'fiscalYear',
'period',
'revenue']])

In [ ]:
# keep only cols we need
revenue_df = df[["symbol", "date", "fiscalYear", "period", "revenue"]].copy()

# cofirm fiscalYear and revenue are numeric
revenue_df['fiscalYear'] = pd.to_numeric(revenue_df['fiscalYear'], errors="coerce")
revenue_df['revenue'] = pd.to_numeric(revenue_df['revenue'], errors="coerce")

# sort oldest to newest
revenue_df = revenue_df.sort_values('fiscalYear')

revenue_df

In [ ]:
# calc rev growth CAGR
start_row = revenue_df.iloc[0]
end_row = revenue_df.iloc[-1]

start_year = start_row['fiscalYear']
end_year = end_row['fiscalYear']

start_revenue = start_row['revenue']
end_revenue = end_row['revenue']

number_of_years = end_year - start_year

rev_growth_cagr = (end_revenue / start_revenue) ** (1 / number_of_years) - 1
rev_growth_cagr_percent = round(rev_growth_cagr * 100, 2)

print(f"Start year: {start_year}")
print(f"End year: {end_year}")
print(f"Start revenue: {start_revenue}")
print(f"End revenue: {end_revenue}")
print(f"Number of years: {number_of_years}")
print(f"Revenue CAGR: {rev_growth_cagr:.4f}")
print(f"Revenue CAGR %: {rev_growth_cagr_percent:.2f}%")
